# Setup

In [5]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


# Defining search space

import torch.nn as nn
from chop.nn.modules import Identity

search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices_idx":[0, 1],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}

# model constructor

from transformers import AutoConfig, AutoModelForSequenceClassification
from chop.tools.utils import deepsetattr


def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)
    # Update the paramaters in the config
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen_idx = trial.suggest_int(param, 0, len(search_space[param]) - 1)
        print(f"upper bound of suggest int = {len(search_space[param]) - 1}")
        print(f"search_space[param] = {search_space[param]}")
        print(f"chosen index = {chosen_idx}")
        setattr(config, param, chosen_idx)

    trial_model = AutoModelForSequenceClassification.from_config(config)
    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == nn.Linear:
                continue
            elif new_layer_cls == Identity:
                new_layer = Identity()
                deepsetattr(trial_model, name, new_layer)
            else:
                raise ValueError(f"Unknown layer type: {new_layer_cls}")

    # for name, layer in trial_model.named_modules():
    #     print(f"name, layer = {name}, {layer}")
    #     if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
    #         print(f"LINEAR LAYER\n\n name, layer = {name}, {layer}\n\n")

    #         # new_layer_cls = trial.suggest_categorical(
    #         #     f"{name}_type",
    #         #     search_space["linear_layer_choices"],
    #         # )
    #         new_layer_cls_idx = trial.suggest_int("linear_layer_choices_idx", 0, 1)
    #         print("\n\n\nNEW LAYER WITH SUGGEST INT: ",new_layer_cls_idx)
    #         print(f"new layer class = {new_layer_cls}")
    #         if new_layer_cls_idx == 0: # linear
    #             continue
    #         elif new_layer_cls == 1: # Identity
    #             new_layer = Identity()
    #             deepsetattr(trial_model, name, new_layer)
    #         else:
    #             raise ValueError(f"Unknown layer type: {new_layer_cls}")
    #         # if new_layer_cls == nn.Linear:
    #         #     continue
    #         # elif new_layer_cls == Identity:
    #         #     new_layer = Identity()
    #         #     deepsetattr(trial_model, name, new_layer)
    #         # else:
    #         #     raise ValueError(f"Unknown layer type: {new_layer_cls}")

    return trial_model

def construct_model(trial):
    config = AutoConfig.from_pretrained(checkpoint)

    # Update the paramaters in the config
    for param in [
        "num_layers",
        "num_heads",
        "hidden_size",
        "intermediate_size",
    ]:
        chosen_idx = trial.suggest_int(param, 0, len(search_space[param]) - 1)
        setattr(config, param, chosen_idx)

    trial_model = AutoModelForSequenceClassification.from_config(config)

    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == nn.Linear:
                continue
            elif new_layer_cls == Identity:
                new_layer = Identity()
                deepsetattr(trial_model, name, new_layer)
            else:
                raise ValueError(f"Unknown layer type: {new_layer_cls}")

    return trial_model


INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6f05572e-0795-4e8e-a3e3-bb4089cc4a24)')' thrown while requesting HEAD https://huggingface.co/datasets/imdb/resolve/main/README.md
Retrying in 1s [Retry 1/5].


# Objective Function

In [6]:

# Trainer
from chop.tools import get_trainer


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    # Set the model as an attribute so we can fetch it later
    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]



# Sampler

In [7]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

import optuna
import joblib



In [8]:


# Random Sampler
def save_checkpoint_random(study, trial):
    joblib.dump(study, "tutorial-5-study-random.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")

study_random = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-random",
    sampler=sampler
)

study_random.optimize(
    objective,
    n_trials=10,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_random],
)


[I 2026-02-02 20:03:08,745] A new study created in memory with name: tutorial-5-study-random
[W 2026-02-02 20:03:08,906] Trial 0 failed with parameters: {'num_layers': 1, 'num_heads': 0, 'hidden_size': 3, 'intermediate_size': 2} because of the following error: ValueError('The hidden size (3) is not a multiple of the number of attention heads (2)').
Traceback (most recent call last):
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_1406/1046848512.py", line 8, in objective
    model = construct_model(trial)
            ^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1406/2114764188.py", line 110, in construct_model
    trial_model = AutoModelForSequenceClassification.from_config(config)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/transformers/mod

ValueError: The hidden size (3) is not a multiple of the number of attention heads (2)

In [9]:
search_space = {
    "num_layers": [2, 4, 8],
    "num_heads": [2, 4, 8, 16],
    "hidden_size": [128, 192, 256, 384, 512],
    "intermediate_size": [512, 768, 1024, 1536, 2048],
    "linear_layer_choices": [
        nn.Linear,
        Identity,
    ],
}

config = AutoConfig.from_pretrained(checkpoint)
trial_model = AutoModelForSequenceClassification.from_config(config)
for name, layer in trial_model.named_modules():
    # print(f"name, layer = {name}, {layer}")
    if isinstance(layer, nn.Linear) and layer.in_features == layer.out_features:
        search_space[f"{name}_type"] = search_space["linear_layer_choices"]
            
# GridSampler
def save_checkpoint_grid(study, trial):
    joblib.dump(study, "tutorial-5-study-grid.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")


sampler = GridSampler(search_space)
study_grid = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-grid",
    sampler=sampler,
)

study_grid.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_grid]
)


/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/samplers/_grid.py:232: UserWarning: linear_layer_choices contains a value with the type of <class 'type'>, which is not supported by `GridSampler`. Please make sure a value is `str`, `int`, `float`, `bool` or `None` for persistent storage.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/samplers/_grid.py:232: UserWarning: bert.encoder.layer.0.attention.self.query_type contains a value with the type of <class 'type'>, which is not supported by `GridSampler`. Please make sure a value is `str`, `int`, `float`, `bool` or `None` for persistent storage.
  warnings.warn(message)
/home/ism/ADL/mase/.venv/lib/python3.11/site-packages/optuna/samplers/_grid.py:232: UserWarning: bert.encoder.layer.0.attention.self.key_type contains a value with the type of <class 'type'>, which is not supported by `GridSampler`. Please make sure a value is `str`, `int`, `float`, `bool` or `None` for persistent storag

Step,Training Loss
500,0.693200
1000,0.641300
1500,0.472800
2000,0.414900
2500,0.388200
3000,0.396700


[I 2026-02-02 20:04:30,813] Trial 0 finished with value: 0.83484 and parameters: {'num_layers': 2, 'num_heads': 16, 'hidden_size': 128, 'intermediate_size': 512, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}. Best is trial 0 wi

Trial 0 finished with value: 0.83484
  Params: {'num_layers': 2, 'num_heads': 16, 'hidden_size': 128, 'intermediate_size': 512, 'bert.encoder.layer.0.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'chop.nn.modules.identity.Identity'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.pooler.dense_type': <class 'chop.nn.modules.identity.Identity'>}
  Model parameters: 4,303,618
  Best so far: 0.83484 

In [ ]:

# TPESampler
def save_checkpoint_tpe(study, trial):
    joblib.dump(study, "tutorial-5-study-tpe.pkl")
    
    model = trial.user_attrs["model"]
    num_params = sum(p.numel() for p in model.parameters())
    
    print(f"Trial {trial.number} finished with value: {trial.value}")
    print(f"  Params: {trial.params}")
    print(f"  Model parameters: {num_params:,}")
    print(f"  Best so far: {study.best_value} (Trial {study.best_trial.number})")

sampler = TPESampler()
study_tpe = optuna.create_study(
    direction="maximize",
    study_name="tutorial-5-study-tpe",
    sampler=sampler,
    callbacks=[save_checkpoint_tpe]
)

study_tpe.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
    callbacks=[save_checkpoint_tpe]
)


TypeError: create_study() got an unexpected keyword argument 'callbacks'

# Save

In [ ]:

# Not sure how to save it at the end
import joblib
joblib.dump(study_random, "./tutorial_5_task_1/study_random.pkl")
joblib.dump(study_grid, "./tutorial_5_task_1/study_grid.pkl")
joblib.dump(study_tpe, "./tutorial_5_task_1/study_tpe.pkl")
